# Sesión 08 — Lab Reto: Pipeline ETL completo + Event Log
## Bronze → Silver → Gold + análisis del event log

**Curso:** Databricks Data Engineer Associate
**Dataset:** `ordenes_logistica.csv` (~80 filas)

---

### Contexto del reto

Construye un pipeline SDP completo para el dominio de logística: una tabla cruda de órdenes en Bronze, una limpia con expectations en Silver, y una agregación de KPIs por transportista en Gold. Luego habilita el event log y analízalo con SQL para responder preguntas operacionales.

### Lo que debes implementar

1. **Bronze streaming table** con Auto Loader (`STREAM read_files`)
2. **Silver streaming table** con tres expectations (drop + warn)
3. **Gold materialized view** con agregaciones por transportista
4. **Habilitar el event log** del pipeline en la configuración
5. **3 queries SQL** sobre el event log para analizar el comportamiento

### Criterios de éxito

- [ ] Bronze tiene ~80 filas
- [ ] Silver tiene < 80 (filas descartadas por expectations)
- [ ] Gold tiene 1 fila por transportista (4 transportistas)
- [ ] El event log muestra al menos 1 expectation con `failed_records > 0`
- [ ] Las queries TODO 4-6 retornan resultados consistentes con el DAG

> Pistas en cada TODO. Lee el enunciado completo antes de empezar.

## Paso 1 — Subir el dataset al Volume

```
/Volumes/dbassociate/default/vol_landing/sesion_08/ordenes/ordenes_logistica.csv
```

Crea la subcarpeta `ordenes/` si no existe.

## Paso 2 — Configuración del Pipeline (UI)

1. Workspace → New → ETL Pipeline
2. **Pipeline name:** `dbassociate_s08_reto`
3. **Source code:** este notebook
4. **Destination:**
   - Catalog: `dbassociate`
   - Default schema: dejar vacío (el código usa full qualified names)
5. **Compute:** Serverless
6. **Mode:** Triggered, Channel: Current
7. **Advanced → Event log** (sección clave del reto):
   - Habilitar Event log
   - Catalog: `dbassociate`
   - Schema: `bronze`
   - Name: `event_log_reto`
8. Save → Start (después de completar los TODOs)

In [0]:
# Importaciones — completar
from pyspark import pipelines as dp
from pyspark.sql import functions as F

## TODO 1 — Bronze streaming table

**Qué hacer:**
- Crear `dbassociate.bronze.ordenes` como streaming table
- Leer con `spark.readStream.format("cloudFiles")` desde `/Volumes/dbassociate/default/vol_landing/sesion_08/ordenes/`
- Configurar `cloudFiles.schemaLocation` en `/Volumes/.../sesion_08/_schemas/bronze_ordenes`
- Agregar columnas `_ingested_at` (timestamp actual) y `_source_file` (`_metadata.file_path`)

**Pista:** sigue el patrón del Lab 1, capa Bronze.

In [0]:
# TODO 1 — Implementar aquí
#
# @dp.table(name="dbassociate.bronze.ordenes")
# def bronze_ordenes():
#     return (
#         spark.readStream
#             .format("cloudFiles")
#             .option("cloudFiles.format", "csv")
#             .option("header", "true")
#             .option("cloudFiles.schemaLocation", ...)
#             .load(...)
#             ...
#     )

## TODO 2 — Silver streaming table con expectations

**Qué hacer:**
- Crear `dbassociate.silver.ordenes_clean` leyendo de `stream(dbassociate.bronze.ordenes)`
- Aplicar tres expectations:
  - `monto_positivo`: `monto > 0` → **DROP ROW**
  - `ciudades_no_nulas`: `ciudad_origen IS NOT NULL AND ciudad_destino IS NOT NULL` → **DROP ROW**
  - `peso_razonable`: `peso_kg < 10000` → **warn only** (`@dp.expect`)
- Cast: `monto` a `decimal(12,2)`, `peso_kg` a `decimal(8,2)`, `fecha` a `date`

**Pista:** usar `@dp.expect_or_drop` para los dos primeros y `@dp.expect` para el tercero.

**Esperado:** ~5 filas se descartan en Silver (2 con monto inválido, 1 con peso negativo, 2 con ciudad nula).

In [0]:
# TODO 2 — Implementar aquí
#
# @dp.table(name="dbassociate.silver.ordenes_clean")
# @dp.expect_or_drop("monto_positivo", "monto > 0")
# @dp.expect_or_drop("ciudades_no_nulas", "ciudad_origen IS NOT NULL AND ciudad_destino IS NOT NULL")
# @dp.expect("peso_razonable", "peso_kg < 10000")
# def silver_ordenes_clean():
#     return (
#         spark.readStream.table("dbassociate.bronze.ordenes")
#             .select(...)
#     )

## TODO 3 — Gold materialized view con KPIs por transportista

**Qué hacer:**
- Crear `dbassociate.gold.kpi_transportistas` como Materialized View (no streaming, recomputa)
- Leer de `dbassociate.silver.ordenes_clean` (sin `stream()` porque es MV)
- Agrupar por `transportista` y calcular:
  - `total_ordenes`: `count(*)`
  - `monto_total`: `sum(monto)`
  - `peso_promedio`: `avg(peso_kg)`
  - `ordenes_entregadas`: `count_if(estado = 'Entregado')`

**Pista:** usar `@dp.materialized_view(name=...)` en lugar de `@dp.table`. Para MV, leer con `spark.read.table(...)` (no readStream).

In [0]:
# TODO 3 — Implementar aquí
#
# @dp.materialized_view(name="dbassociate.gold.kpi_transportistas")
# def gold_kpi_transportistas():
#     return (
#         spark.read.table("dbassociate.silver.ordenes_clean")
#             .groupBy("transportista")
#             .agg(...)
#     )

## Paso 3 — Ejecutar el pipeline

Vuelve a la UI del pipeline y pulsa **Start**. Verifica:
- DAG con 3 cajas: bronze.ordenes → silver.ordenes_clean → gold.kpi_transportistas
- Todas en estado COMPLETED (verde)
- Métricas: bronze ~80 filas, silver < 80, gold 4 filas (una por transportista)

Si algún TODO está incompleto, el pipeline fallará. El event log mostrará el error específico.

## Análisis del event log (TODO 4-6)

Las siguientes queries SE EJECUTAN EN SQL EDITOR (no aquí). Reemplaza los `?` con los filtros correctos.

El event log se accede con `event_log(TABLE(...))` cuando está publicado como tabla en UC.

### TODO 4 — Filas procesadas por flow en cada update

```sql
-- Pista: filtrar event_type = 'flow_progress' con status COMPLETED
-- Extraer details:flow_progress.metrics.num_output_rows

SELECT
    timestamp,
    origin.flow_name,
    details:flow_progress.metrics.num_output_rows AS rows_out
FROM event_log(TABLE(?))
WHERE event_type = '?'
    AND details:flow_progress.status = '?'
ORDER BY timestamp DESC;
```

### TODO 5 — Expectations: filas pasadas y descartadas

```sql
-- Pista: el campo details:flow_progress.data_quality.expectations es un array JSON.
-- Hay que parsearlo con from_json y luego explode.

SELECT
    origin.flow_name,
    expectations.name,
    expectations.dataset,
    expectations.passed_records,
    expectations.failed_records
FROM (
    SELECT
        origin,
        explode(from_json(
            details:flow_progress.data_quality.expectations,
            'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>'
        )) AS expectations
    FROM event_log(TABLE(?))
    WHERE event_type = '?'
);
```

### TODO 6 — Lineage del Gold

```sql
-- Pista: event_type = 'flow_definition' contiene el output_dataset y los input_datasets

SELECT
    details:flow_definition.output_dataset AS output_table,
    details:flow_definition.input_datasets AS input_tables
FROM event_log(TABLE(?))
WHERE event_type = '?'
    AND details:flow_definition.output_dataset LIKE '%kpi_transportistas%';
```

## Validación final

Marca cada criterio cuando se cumpla:

- [ ] `SELECT COUNT(*) FROM dbassociate.bronze.ordenes` retorna 80
- [ ] `SELECT COUNT(*) FROM dbassociate.silver.ordenes_clean` retorna entre 73 y 78 (algunas descartadas)
- [ ] `SELECT COUNT(*) FROM dbassociate.gold.kpi_transportistas` retorna 4
- [ ] La query TODO 4 muestra al menos 3 filas (una por flow: bronze, silver, gold)
- [ ] La query TODO 5 muestra al menos una expectation con `failed_records > 0`
- [ ] La query TODO 6 muestra que `kpi_transportistas` depende de `silver.ordenes_clean`

### Pregunta bonus

Ejecuta el pipeline una segunda vez sin agregar nuevos archivos. ¿Cuántas filas procesa Bronze en la segunda ejecución? ¿Y Silver? ¿Y Gold? Justifica usando lo que sabes de Streaming Tables vs Materialized Views.

In [0]:
# LIMPIEZA — ejecutar en SQL editor
#
# DROP TABLE IF EXISTS dbassociate.bronze.ordenes;
# DROP TABLE IF EXISTS dbassociate.silver.ordenes_clean;
# DROP TABLE IF EXISTS dbassociate.gold.kpi_transportistas;
# DROP TABLE IF EXISTS dbassociate.bronze.event_log_reto;
# -- Borrar el pipeline desde la UI
# dbutils.fs.rm("/Volumes/dbassociate/default/vol_landing/sesion_08/_schemas/bronze_ordenes", recurse=True)